# NBA Salary Prediction — Modeling

Goal: Train and compare regression models starting with the 
simplest first. Complexity only added when simpler models fall short.

The progression follows the bias-variance tradeoff:
1. High bias, low variance → Linear Regression → "What is the simplest signal?"
2. Lower bias, controlled variance → Random Forest → "What nonlinear structure exists?"
3. Low bias, optimized variance → XGBoost → "How far can we push performance?"

Target variable: log_salary — predictions converted back to dollars using exp()
Evaluation metrics: RMSE (in dollars) and R²

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from pathlib import Path
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor

In [8]:
DATA_DIR = Path('../data')

X_train = pd.read_pickle(DATA_DIR / 'X_train.pkl')
X_test  = pd.read_pickle(DATA_DIR / 'X_test.pkl')
y_train = pd.read_pickle(DATA_DIR / 'y_train.pkl')
y_test  = pd.read_pickle(DATA_DIR / 'y_test.pkl')

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")

X_train: (328, 19)
X_test:  (82, 19)


## Evaluation helper function

Consistent metric calculation across all models.
RMSE converted back to dollars using exp() for interpretability —
"on average the model's salary prediction is off by $X million"

In [9]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    # Predictions
    train_pred = model.predict(X_train)
    test_pred  = model.predict(X_test)
    
    # R² scores
    train_r2 = r2_score(y_train, train_pred)
    test_r2  = r2_score(y_test, test_pred)
    
    # RMSE in log space
    train_rmse_log = np.sqrt(mean_squared_error(y_train, train_pred))
    test_rmse_log  = np.sqrt(mean_squared_error(y_test, test_pred))
    
    # RMSE converted back to dollars
    train_rmse_usd = np.sqrt(mean_squared_error(
        np.exp(y_train), np.exp(train_pred)))
    test_rmse_usd  = np.sqrt(mean_squared_error(
        np.exp(y_test), np.exp(test_pred)))
    
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  Train R²:   {train_r2:.3f}")
    print(f"  Test R²:    {test_r2:.3f}")
    print(f"  Train RMSE: ${train_rmse_usd/1e6:.1f}M")
    print(f"  Test RMSE:  ${test_rmse_usd/1e6:.1f}M")
    
    return {
        'model': name,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'train_rmse_usd': train_rmse_usd,
        'test_rmse_usd': test_rmse_usd,
        'predictions': test_pred
    }


## Model 1: Linear Regression — baseline

Simplest possible model — no regularization, default settings.
Establishes the baseline performance before adding complexity.

In [10]:
lr = LinearRegression()
lr.fit(X_train, y_train)

lr_results = evaluate_model(
    'Linear Regression', lr, X_train, X_test, y_train, y_test
)


  Linear Regression
  Train R²:   0.487
  Test R²:    0.539
  Train RMSE: $11.2M
  Test RMSE:  $8.7M


## Linear Regression baseline results

R²: 0.539 — model explains 54% of salary variance
RMSE: $8.7M — average prediction error

Test R² slightly higher than Train R² (0.539 vs 0.487) — 
unusual but expected with a small test set of 82 players.
Not a sign of data leakage.

The $8.7M RMSE reflects a fundamental limitation: stats alone 
cannot capture contract timing, market dynamics, or perceived 
upside. This is our baseline to beat.

Before moving to Random Forest, we try Ridge and Lasso regression —
regularized versions of linear regression that penalize large 
coefficients. These often outperform vanilla linear regression 
on datasets with multicollinearity.

In [11]:
alphas = [0.01, 0.1, 1, 10, 100, 1000]
ridge_scores = []

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    scores = cross_val_score(ridge, X_train, y_train, 
                             cv=5, scoring='r2')
    ridge_scores.append(scores.mean())
    print(f"Ridge alpha={alpha:>6}: CV R² = {scores.mean():.3f}")

best_alpha = alphas[np.argmax(ridge_scores)]
print(f"\nBest alpha: {best_alpha}")

Ridge alpha=  0.01: CV R² = 0.412
Ridge alpha=   0.1: CV R² = 0.413
Ridge alpha=     1: CV R² = 0.413
Ridge alpha=    10: CV R² = 0.415
Ridge alpha=   100: CV R² = 0.397
Ridge alpha=  1000: CV R² = 0.216

Best alpha: 10


In [13]:
ridge = Ridge(alpha=10)
ridge.fit(X_train, y_train)

ridge_results = evaluate_model(
    'Ridge Regression', ridge, X_train, X_test, y_train, y_test
)


  Ridge Regression
  Train R²:   0.486
  Test R²:    0.540
  Train RMSE: $10.9M
  Test RMSE:  $8.8M


## Model 3: Lasso Regression

Lasso adds L1 regularization — can shrink coefficients to exactly 
zero, performing automatic feature selection.
Useful for identifying which stats truly matter vs noise.

In [12]:
lasso_scores = []
lasso_alphas = [0.001, 0.01, 0.1, 1, 10]

for alpha in lasso_alphas:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    scores = cross_val_score(lasso, X_train, y_train,
                             cv=5, scoring='r2')
    lasso_scores.append(scores.mean())
    print(f"Lasso alpha={alpha:>5}: CV R² = {scores.mean():.3f}")

best_lasso_alpha = lasso_alphas[np.argmax(lasso_scores)]
print(f"\nBest alpha: {best_lasso_alpha}")

lasso = Lasso(alpha=best_lasso_alpha, max_iter=10000)
lasso.fit(X_train, y_train)

lasso_results = evaluate_model(
    'Lasso Regression', lasso, X_train, X_test, y_train, y_test
)

# Show which features Lasso zeroed out
zero_features = [f for f, c in zip(X_train.columns, lasso.coef_) if c == 0]
print(f"\nFeatures zeroed out by Lasso: {zero_features}")

Lasso alpha=0.001: CV R² = 0.413
Lasso alpha= 0.01: CV R² = 0.417
Lasso alpha=  0.1: CV R² = 0.364
Lasso alpha=    1: CV R² = -0.026
Lasso alpha=   10: CV R² = -0.026

Best alpha: 0.01

  Lasso Regression
  Train R²:   0.484
  Test R²:    0.544
  Train RMSE: $11.0M
  Test RMSE:  $8.7M

Features zeroed out by Lasso: ['3P%', 'eFG%', 'DRB', 'STL', 'TOV', 'Pos_SF']


## Lasso findings — automatic feature selection

Lasso zeroed out 6 features:
- 3P%: three point shooting % has no independent salary signal
- eFG%: shooting efficiency not directly rewarded by contracts
- DRB: defensive rebounds redundant with overall defensive profile
- STL: steals not independently valued in contracts
- TOV: turnovers redundant with high usage stats already present
- Pos_SF: small forward position has no salary premium vs center

This confirms EDA finding that shooting efficiency metrics 
(eFG%, 3P%) barely correlate with salary (0.05-0.07).
The NBA pays for volume and usage, not efficiency.

Lasso performance:
- Test R²: 0.544 — marginally better than Linear Regression (0.539)
- Test RMSE: $8.7M — identical to Linear Regression

Regularization provided minimal improvement — suggests the 
multicollinearity in our features is not severely hurting 
linear regression performance.

## Linear model comparison

| Model | Train R² | Test R² | Train RMSE | Test RMSE |
|---|---|---|---|---|
| Linear Regression | 0.487 | 0.539 | $11.2M | $8.7M |
| Ridge (alpha=10) | 0.486 | 0.540 | $10.9M | $8.8M |
| Lasso (alpha=0.01) | 0.484 | 0.544 | $11.0M | $8.7M |

All three linear models perform nearly identically — R² ~0.54 
and RMSE ~$8.7M. Regularization provided negligible improvement 
over vanilla linear regression.

This tells us two things:
1. Multicollinearity is not severely hurting linear performance
2. The relationship between stats and salary has nonlinear 
   components that linear models cannot capture

Lasso is our best linear model at Test R² 0.544.
This is our baseline to beat with Random Forest.

In [14]:
rf = RandomForestRegressor(random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_results = evaluate_model(
    'Random Forest', rf, X_train, X_test, y_train, y_test
)


  Random Forest
  Train R²:   0.920
  Test R²:    0.496
  Train RMSE: $4.5M
  Test RMSE:  $9.6M


## Random Forest default — overfitting detected

Train R²: 0.920 vs Test R²: 0.496 — gap of 0.424
Train RMSE: $4.5M vs Test RMSE: $9.6M

Classic overfitting. Default Random Forest grows deep trees 
with no constraints — it memorizes training data perfectly 
but fails to generalize. Test RMSE of $9.6M is actually 
worse than our linear baseline of $8.7M.

Solution: Constrain the trees using max_depth and 
min_samples_leaf to force the model to learn general 
patterns rather than memorizing training examples.

In [ ]:
rf_tuned = RandomForestRegressor(
    n_estimators=200,
    max_depth=6,
    min_samples_leaf=5,
    max_features=0.7,
    random_state=42,
    n_jobs=-1
)
rf_tuned.fit(X_train, y_train)

rf_tuned_results = evaluate_model(
    'Random Forest (tuned)', rf_tuned, 
    X_train, X_test, y_train, y_test
)


  Random Forest (tuned)
  Train R²:   0.713
  Test R²:    0.505
  Train RMSE: $7.6M
  Test RMSE:  $9.8M


In [16]:
rf_tuned2 = RandomForestRegressor(
    n_estimators=500,
    max_depth=4,
    min_samples_leaf=10,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)
rf_tuned2.fit(X_train, y_train)

rf_tuned2_results = evaluate_model(
    'Random Forest (tuned v2)', rf_tuned2,
    X_train, X_test, y_train, y_test
)


  Random Forest (tuned v2)
  Train R²:   0.522
  Test R²:    0.409
  Train RMSE: $10.7M
  Test RMSE:  $11.6M


In [17]:
rf_tuned3 = RandomForestRegressor(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=8,
    max_features=0.8,
    random_state=42,
    n_jobs=-1
)
rf_tuned3.fit(X_train, y_train)

rf_tuned3_results = evaluate_model(
    'Random Forest (tuned v3)', rf_tuned3,
    X_train, X_test, y_train, y_test
)


  Random Forest (tuned v3)
  Train R²:   0.633
  Test R²:    0.500
  Train RMSE: $8.4M
  Test RMSE:  $9.9M


## Random Forest — conclusion

All Random Forest variants tested:

| Model | Train R² | Test R² | Test RMSE |
|---|---|---|---|
| Default | 0.920 | 0.496 | $9.6M |
| Tuned v1 (depth=6) | 0.713 | 0.505 | $9.8M |
| Tuned v2 (depth=4) | 0.522 | 0.409 | $11.6M |
| Tuned v3 (depth=5) | 0.633 | 0.500 | $9.9M |

Random Forest consistently underperforms linear models on this 
dataset. Best test RMSE is $9.6M vs Lasso's $8.7M.

Root cause: 328 training samples is a small dataset for Random 
Forest. Ensemble methods need more data to build diverse trees 
that generalize well. With limited samples, trees see overlapping 
subsets and learn similar patterns — the ensemble advantage 
disappears.

Moving to XGBoost — gradient boosting builds trees sequentially, 
correcting previous errors. It often outperforms Random Forest 
on small-to-medium datasets.

In [18]:
xgb = XGBRegressor(
    random_state=42,
    n_jobs=-1,
    eval_metric='rmse'
)
xgb.fit(X_train, y_train)

xgb_results = evaluate_model(
    'XGBoost default', xgb,
    X_train, X_test, y_train, y_test
)


  XGBoost default
  Train R²:   1.000
  Test R²:    0.333
  Train RMSE: $0.0M
  Test RMSE:  $11.1M


## XGBoost default — severe overfitting

Train R²: 1.000 vs Test R²: 0.333 — gap of 0.667
XGBoost default settings grow unconstrained trees that perfectly 
memorize training data. Needs aggressive regularization.

In [19]:
xgb_tuned = XGBRegressor(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=5.0,
    min_child_weight=5,
    random_state=42,
    n_jobs=-1,
    eval_metric='rmse'
)
xgb_tuned.fit(X_train, y_train)

xgb_tuned_results = evaluate_model(
    'XGBoost tuned', xgb_tuned,
    X_train, X_test, y_train, y_test
)


  XGBoost tuned
  Train R²:   0.838
  Test R²:    0.512
  Train RMSE: $6.1M
  Test RMSE:  $9.8M


In [20]:
xgb_tuned2 = XGBRegressor(
    n_estimators=100,
    max_depth=2,
    learning_rate=0.05,
    subsample=0.7,
    colsample_bytree=0.6,
    reg_alpha=5.0,
    reg_lambda=10.0,
    min_child_weight=10,
    random_state=42,
    n_jobs=-1,
    eval_metric='rmse'
)
xgb_tuned2.fit(X_train, y_train)

xgb_tuned2_results = evaluate_model(
    'XGBoost tuned v2', xgb_tuned2,
    X_train, X_test, y_train, y_test
)


  XGBoost tuned v2
  Train R²:   0.568
  Test R²:    0.504
  Train RMSE: $9.4M
  Test RMSE:  $9.9M


## Final model comparison — all models

| Model | Train R² | Test R² | Test RMSE |
|---|---|---|---|
| Linear Regression | 0.487 | 0.539 | $8.7M |
| Ridge (alpha=10) | 0.486 | 0.540 | $8.8M |
| Lasso (alpha=0.01) | 0.484 | 0.544 | $8.7M |
| Random Forest default | 0.920 | 0.496 | $9.6M |
| Random Forest tuned v1 | 0.713 | 0.505 | $9.8M |
| XGBoost default | 1.000 | 0.333 | $11.1M |
| XGBoost tuned v1 | 0.838 | 0.512 | $9.8M |
| XGBoost tuned v2 | 0.568 | 0.504 | $9.9M |

Winner: Lasso Regression (alpha=0.01)
- Test R²:   0.544
- Test RMSE: $8.7M

The simplest model won.

This is a meaningful finding not a disappointing one:
- 328 training samples is too small for ensemble methods 
  to build diverse generalizable trees
- NBA salary has largely linear relationships with stats 
  as confirmed by our correlation analysis
- Tree-based models overfit aggressively on small datasets 
  regardless of regularization attempts
- Lasso's automatic feature selection (zeroing out 6 features) 
  produced a clean, interpretable model

The $8.7M RMSE reflects an irreducible error — stats alone 
cannot capture contract timing, market dynamics, age curves, 
and perceived upside. This is expected and honest.